## Evaluation for Disambiguation
The evaluation for disambiguation will be split up into three stages, with the first and the last done in this notebook:
1) Parsing the evaluation set
2) Creating the gold standard from the eval set (done by hand)
3) Evaluating the model on the gold standard

### Parsing the evaluation set
For our evaluation set, we will be using a sample of 4646 example sentences from the OPD, of which we will sample sentences that are fully analyzed by the FST (but could contain no ambiguities). 

Here is the code for that:

In [5]:

# setup and imports
import csv, sys, os, random
from pathlib import Path

sys.path.append(os.path.abspath(".."))
from src.disambiguation import (
    load_fst_parser,
    tokenize,
    fst_parse_sentence,
    ojibwe_sentence_to_cg3_format,
    PUNCTUATIONS,
    PRESERVE_TOKEN,
)


In [6]:
# defining variables and paths
SEED = 421
OUTDIR = Path("eval_data")
OUTDIR.mkdir(parents=True, exist_ok=True)
TSV_PATH = "../data/eval/example_sentences.tsv"     
FST_BINARY_PATH = "../data/fst/ojibwe7.fomabin" 

In [7]:
# helpers to build eval set

def is_word_token(tok: str) -> bool:
    """Everything other than punctuation and ellipsis is a word token"""
    return tok not in set(PUNCTUATIONS) and tok != PRESERVE_TOKEN

def all_nonpunct_tokens_parsed(oj: str, fst) -> bool:
    """True iff every non-punctuation token has at least one FST analysis."""
    toks = tokenize(oj)
    analyses = fst_parse_sentence(toks, fst)
    for item in analyses:
        tok = item["word_form"]
        if not is_word_token(tok):
            continue
        if len(item["fst_analyses"]) == 0:
            return False
    return True

# main read / write functions
def read_opd_tsv(tsv_path: str):
    """Return list of dicts w/ Ojibwe, English, Speaker, Link from a TSV."""
    rows = []
    with open(tsv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader, None)
        # skip header if it looks like the column names
        if header and all(h.lower() in ("ojibwe","english","speaker","link") for h in header):
            pass
        else:
            # header is actually data
            if header:
                cells = (header + ["", "", "", ""])[:4]
                rows.append({
                    "Ojibwe": cells[0].strip(),
                    "English": cells[1].strip(),
                    "Speaker": cells[2].strip(),
                    "Link": cells[3].strip(),
                })

        for row in reader:
            if not row or all(not x.strip() for x in row):
                continue
            cells = (row + ["", "", "", ""])[:4]
            rows.append({
                "Ojibwe": cells[0].strip(),
                "English": cells[1].strip(),
                "Speaker": cells[2].strip(),
                "Link": cells[3].strip(),
            })
    return rows


# make corresponding tsv just in case
def write_tsv(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t")
        w.writerow(["sent_id", "Ojibwe", "English", "Speaker", "Link"])
        for i, r in enumerate(rows, 1):
            w.writerow([i, r["Ojibwe"], r["English"], r["Speaker"], r["Link"]])

# write the main .txt file used as the gold standard
def write_cg3(rows, fst, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows, 1):
            cg3 = ojibwe_sentence_to_cg3_format(r["Ojibwe"], fst)
            f.write(f"# sent_id = {i}\n# text = {r['Ojibwe']}\n# eng = {r['English']}\n")
            f.write(cg3)


In [8]:

# main code to parse the input csv

# load fst
print(f"Loading FST from: {FST_BINARY_PATH}")
fst = load_fst_parser(FST_BINARY_PATH)
# read csv
print(f"Reading OPD TSV: {TSV_PATH}")
all_rows = read_opd_tsv(TSV_PATH)
print(f"Total good rows in TSV: {len(all_rows)}")

# for reproducibility
import random
rng = random.Random(SEED)
rng.shuffle(all_rows)

sample_size = 300
print(f"Filtering {sample_size} rows with all tokens parsed by FST")
keep = []
for r in all_rows:
    oj = r.get("Ojibwe", "").strip()
    if not oj:
        continue
    try:
        if all_nonpunct_tokens_parsed(oj, fst):
            keep.append(r)
        # stop at 300 sentences
        if len(keep) >= sample_size:
            break
    except Exception as e:
        # skip on exceptions
        continue

print(f"Number rows parsed: {len(keep)}")


Loading FST from: ../data/fst/ojibwe7.fomabin
FST file is ../data/fst/ojibwe7.fomabin
Reading OPD TSV: ../data/eval/example_sentences.tsv
Total good rows in TSV: 4645
Filtering 300 rows with all tokens parsed by FST
Number rows parsed: 300


In [9]:

# make tsv and cg3 formatted files with the parsed sentences
# write a file with all 300 lines and another with 100 (in case 300 is too many)
write_tsv(keep[:100], OUTDIR / "sample_100.tsv")
write_cg3(keep[:100], fst, OUTDIR / "sample_100.txt")
write_tsv(keep, OUTDIR / "sample_300.tsv")
write_cg3(keep, fst, OUTDIR / "sample_300.txt")

print("Wrote files:")
print(OUTDIR / "sample_100.tsv")
print(OUTDIR / "sample_100.txt")
print(OUTDIR / "sample_300.tsv")
print(OUTDIR / "sample_300.txt")


Wrote files:
eval_data/sample_100.tsv
eval_data/sample_100.txt
eval_data/sample_300.tsv
eval_data/sample_300.txt


### Creating the gold standard

The gold standard was created by doing disambiguation by hand on the sample sets parsed above. Unwanted lines were deleted from the .cg3 file by hand, as if the human was the disambiguation module. 

### Evaluating the model on the gold standard

Will be done once the gold standard is ready. 

In [2]:
# Reparse 100 sentences listed in a previous CG3 file (grabbed after "# text = ...")

import os, sys, re, csv
from pathlib import Path

# --- config (adjust paths if needed) ---
INPUT_CG3_PATH   = Path("eval_data/opd_sample_100.txt")     # the file you pasted from
FST_BINARY_PATH  = "../data/fst/ojibwe7.fomabin"            # your FST
OUTDIR           = Path("eval_data")
OUT_CG3_PATH     = OUTDIR / "opd_sample_100_reparsed.txt"
OUT_TSV_PATH     = OUTDIR / "opd_sample_100_reparsed.tsv"

# --- wire up your codebase helpers ---
sys.path.append(os.path.abspath(".."))
from src.disambiguation import (
    load_fst_parser,
    ojibwe_sentence_to_cg3_format,
)

def read_texts_from_cg3(cg3_path: Path):
    """
    Return list of dicts with fields:
      - sent_id (int or None)
      - Ojibwe (str)   <- value from '# text = ...'
      - English (str or '')  <- value from '# eng = ...' if present
    Reads in order, keeps ALL 100 (does not validate with FST).
    """
    rows = []
    sent_id = None
    oj = None
    eng = ""

    rx_id   = re.compile(r"^#\s*sent_id\s*=\s*(\d+)\s*$")
    rx_text = re.compile(r"^#\s*text\s*=\s*(.*)\s*$")
    rx_eng  = re.compile(r"^#\s*eng\s*=\s*(.*)\s*$")

    with open(cg3_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            m_id = rx_id.match(line)
            m_tx = rx_text.match(line)
            m_en = rx_eng.match(line)
            if m_id:
                # starting a new block; if we had a previous complete block, store it
                if oj is not None:
                    rows.append({"sent_id": sent_id, "Ojibwe": oj, "English": eng})
                    oj, eng = None, ""
                sent_id = int(m_id.group(1))
            elif m_tx:
                oj = m_tx.group(1).strip()
            elif m_en:
                eng = m_en.group(1).strip()

        # flush last block
        if oj is not None:
            rows.append({"sent_id": sent_id, "Ojibwe": oj, "English": eng})

    return rows

def write_cg3(rows, fst, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows, 1):
            cg3 = ojibwe_sentence_to_cg3_format(r["Ojibwe"], fst)
            # keep original eng if present (may be empty)
            eng = r.get("English", "")
            f.write(f"# sent_id = {i}\n# text = {r['Ojibwe']}\n# eng = {eng}\n")
            f.write(cg3)

def write_tsv(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t")
        w.writerow(["sent_id", "Ojibwe", "English"])
        for i, r in enumerate(rows, 1):
            w.writerow([i, r["Ojibwe"], r.get("English", "")])

# --- run ---
print(f"Loading FST from: {FST_BINARY_PATH}")
fst = load_fst_parser(FST_BINARY_PATH)

print(f"Reading CG3 source: {INPUT_CG3_PATH}")
rows_100 = read_texts_from_cg3(INPUT_CG3_PATH)
print(f"Found rows: {len(rows_100)} (expected ~100)")

OUTDIR.mkdir(parents=True, exist_ok=True)
write_cg3(rows_100, fst, OUT_CG3_PATH)
write_tsv(rows_100, OUT_TSV_PATH)

print("Wrote:")
print(" ", OUT_CG3_PATH)
print(" ", OUT_TSV_PATH)


Loading FST from: ../data/fst/ojibwe7.fomabin
FST file is ../data/fst/ojibwe7.fomabin
Reading CG3 source: eval_data/opd_sample_100.txt
Found rows: 100 (expected ~100)
Wrote:
  eval_data/opd_sample_100_reparsed.txt
  eval_data/opd_sample_100_reparsed.tsv


In [4]:
import re
from pathlib import Path

# --- OPTION A: paste your text block here ---
raw = None  # replace with your multiline string, e.g. raw = """...your text..."""

# --- OPTION B: or point to a file (leave as None to skip) ---
path = Path("eval_data/opd_sample_100_reparsed.txt")  # e.g., Path("eval_data/opd_sample_100_reparsed.txt")

# load text
if raw is not None:
    text = raw
elif path is not None:
    text = Path(path).read_text(encoding="utf-8")
else:
    raise ValueError("Provide data: set `raw` to the pasted text OR set `path` to a file.")

sent_re = re.compile(r'^#\s*sent_id\s*=\s*(\d+)\s*$', re.M)
has_chcnj = False
current_id = None
hits = []

for line in text.splitlines():
    m = sent_re.match(line)
    if m:
        if current_id is not None and has_chcnj:
            hits.append(current_id)
        current_id = int(m.group(1))
        has_chcnj = False
        continue
    if "ChCnj" in line.split():  # simple token check
        has_chcnj = True

# flush last sentence
if current_id is not None and has_chcnj:
    hits.append(current_id)

print(hits), len(hits)


[1, 4, 6, 16, 26, 29, 32, 37, 45, 46, 49, 52, 54, 55, 57, 59, 61, 64, 65, 72, 81, 84, 85, 89]


(None, 24)

In [ ]:
from pathlib import Path
import sys, os
# --- config (adjust paths if needed) ---
FST_BINARY_PATH  = "../data/fst/ojibwe7.fomabin"            # your FST

# --- wire up your codebase helpers ---
sys.path.append(os.path.abspath(".."))
from src.disambiguation import (
    load_fst_parser,
    ojibwe_sentence_to_cg3_format,
    fst_parse_word
)

fst = load_fst_parser(FST_BINARY_PATH)
fst_parse_word("mii", fst)


FST file is ../data/fst/ojibwe7.fomabin


['mii+ADVPred']